# 🔍 Fake Review Forensic Analyzer — DistilBERT Training

> Fine-tuning DistilBERT on the **Kaggle Amazon Fake Reviews Dataset** (~21,000 real reviews).

### ⚡ Before Running:
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Make sure you have copied your **Kaggle API Token** to your clipboard.

### Steps:
1. Setup Kaggle API (Paste your token)
2. Download & explore dataset
3. Preprocess data
4. Fine-tune DistilBERT (~15 min on T4 GPU)
5. Evaluate model
6. Download trained model to your PC


## ⚡ Step 0 — Check GPU
> Make sure GPU is enabled: **Runtime → Change runtime type → T4 GPU**

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected!')
    print('Go to Runtime → Change runtime type → T4 GPU and restart.')


## 📦 Step 1 — Setup Kaggle API

Run the cell below and **paste the Kaggle API Token** you copied from the Kaggle settings page.


In [ ]:
import os
from getpass import getpass

print('🔑 Please paste your Kaggle API Token below and press Enter:')
token = getpass('Token: ')
os.environ['KAGGLE_API_TOKEN'] = token.strip()
print('✅ Kaggle API configured successfully!')


## 📦 Step 2 — Install Dependencies

In [ ]:
# Install required packages — do NOT touch sympy (leave it for torch)
!pip install -q --upgrade transformers
!pip install -q kaggle scikit-learn pandas numpy lime plotly joblib
print('✅ All packages installed!')
print()
print('⚠️  IMPORTANT: You MUST restart the session now!')
print('   Go to Runtime → Restart session → then Runtime → Run all again.')
import os
os.kill(os.getpid(), 9)  # Auto-restarts the kernel


## 📊 Step 3 — Download Kaggle Dataset
> Dataset: **Amazon Fake Reviews** by mexwell (~21,000 reviews)
> Columns: `text_` (review text), `label` (CG=Fake, OR=Original/Real)


In [ ]:
# Download dataset from Kaggle
!kaggle datasets download -d mexwell/fake-reviews-dataset --unzip

import os, glob
csv_files = glob.glob('*.csv')
print('Downloaded files:', csv_files)


## 🔍 Step 4 — Explore & Preprocess Dataset

In [ ]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('fake reviews dataset.csv')
print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nFirst few rows:')
print(df.head())
print('\nLabel distribution:')
print(df['label'].value_counts())


In [ ]:
# Preprocess
# label: CG = Computer Generated (FAKE=1), OR = Original Real (AUTHENTIC=0)
df = df[['text_', 'label']].dropna()
df.columns = ['review', 'label_raw']
df['label'] = df['label_raw'].map({'CG': 1, 'OR': 0})
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

# Remove very short reviews (< 20 chars)
df = df[df['review'].str.len() > 20].reset_index(drop=True)

# Balance dataset (equal fake and authentic)
min_count = min(df['label'].value_counts())
df = pd.concat([
    df[df['label'] == 0].sample(min_count, random_state=42),
    df[df['label'] == 1].sample(min_count, random_state=42),
]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'✅ Final dataset: {len(df)} samples')
print(f'   Fake (CG)      : {(df["label"]==1).sum()}')
print(f'   Authentic (OR) : {(df["label"]==0).sum()}')
print(f'\nSample fake review:')
print(df[df['label']==1]['review'].iloc[0][:300])
print(f'\nSample authentic review:')
print(df[df['label']==0]['review'].iloc[0][:300])

df.to_csv('reviews_dataset.csv', index=False)
print('\n✅ Saved as reviews_dataset.csv')


## 🚀 Step 5 — Fine-Tune DistilBERT
> ⏳ Expected time: **~15-20 minutes** on T4 GPU


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
import joblib, os

# ── Config ───────────────────────────────────────────────────────────────────
MODEL_NAME    = 'distilbert-base-uncased'
MAX_LEN       = 256
BATCH_SIZE    = 16
EPOCHS        = 3
LEARNING_RATE = 2e-5
SAVE_DIR      = 'bert_model'
DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

# ── Dataset class ─────────────────────────────────────────────────────────────
class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long),
        }

# ── Load data ─────────────────────────────────────────────────────────────────
df     = pd.read_csv('reviews_dataset.csv')
texts  = df['review'].tolist()
labels = df['label'].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels)
X_train, X_val, y_train, y_val   = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42, stratify=y_train)

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

# ── Tokenizer & Loaders ───────────────────────────────────────────────────────
print('\n📥 Loading DistilBERT tokenizer...')
tokenizer    = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
train_loader = DataLoader(ReviewDataset(X_train, y_train, tokenizer, MAX_LEN),
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(ReviewDataset(X_val,   y_val,   tokenizer, MAX_LEN),
                          batch_size=BATCH_SIZE)
test_loader  = DataLoader(ReviewDataset(X_test,  y_test,  tokenizer, MAX_LEN),
                          batch_size=BATCH_SIZE)

# ── Model, optimizer, scheduler ───────────────────────────────────────────────
print('📥 Loading DistilBERT model...')
model     = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)

# ── Helper: one epoch ─────────────────────────────────────────────────────────
def run_epoch(model, loader, optimizer=None, scheduler=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, preds_all, labels_all, probs_all = 0, [], [], []
    with torch.set_grad_enabled(is_train):
        for batch in loader:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            lbls = batch['label'].to(DEVICE)
            out  = model(input_ids=ids, attention_mask=mask, labels=lbls)
            if is_train:
                out.loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()
            total_loss += out.loss.item()
            preds_all.extend(torch.argmax(out.logits, 1).cpu().numpy())
            labels_all.extend(lbls.cpu().numpy())
            probs_all.extend(torch.softmax(out.logits, 1)[:, 1].cpu().numpy())
    acc = accuracy_score(labels_all, preds_all)
    auc = roc_auc_score(labels_all, probs_all)
    return total_loss / len(loader), acc, auc, labels_all, preds_all

# ── Training Loop ─────────────────────────────────────────────────────────────
best_val_acc = 0
print(f'\n🚀 Training for {EPOCHS} epochs on {DEVICE}...')
print('-' * 70)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc, tr_auc, _, _ = run_epoch(model, train_loader, optimizer, scheduler)
    vl_loss, vl_acc, vl_auc, _, _ = run_epoch(model, val_loader)
    print(f'Epoch {epoch}/{EPOCHS} | '
          f'Train → Loss:{tr_loss:.4f} Acc:{tr_acc:.4f} | '
          f'Val → Loss:{vl_loss:.4f} Acc:{vl_acc:.4f} AUC:{vl_auc:.4f}')
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        os.makedirs(SAVE_DIR, exist_ok=True)
        model.save_pretrained(SAVE_DIR)
        tokenizer.save_pretrained(SAVE_DIR)
        print(f'  ✅ Best model checkpoint saved → {SAVE_DIR}/')

print('\n🎉 Training complete!')


## 📊 Step 6 — Evaluate on Test Set

In [ ]:
from transformers import DistilBertForSequenceClassification

# Load best saved model
best_model = DistilBertForSequenceClassification.from_pretrained(SAVE_DIR)
best_model.to(DEVICE)

_, test_acc, test_auc, y_true, y_pred = run_epoch(best_model, test_loader)

print('=' * 50)
print(f'  Final Test Results')
print('=' * 50)
print(f'  Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)')
print(f'  ROC-AUC  : {test_auc:.4f}')
print()
print(classification_report(y_true, y_pred, target_names=['Authentic', 'Fake']))

# Save metadata
import joblib
meta = {
    'accuracy':   test_acc,
    'auc':        test_auc,
    'model_type': 'DistilBERT',
    'dataset':    'Kaggle Amazon Fake Reviews (~21,000 samples)',
    'classes':    ['Authentic', 'Fake'],
}
joblib.dump(meta, 'model_meta.pkl')
print('✅ model_meta.pkl saved!')


## 📥 Step 7 — Download Trained Model to Your PC
> This zips the entire `bert_model/` folder and downloads it automatically.


In [ ]:
import shutil
from google.colab import files

# Copy metadata into the model folder
shutil.copy('model_meta.pkl', f'{SAVE_DIR}/model_meta.pkl')

# Zip the folder
shutil.make_archive('bert_model_download', 'zip', '.', SAVE_DIR)
print('✅ bert_model_download.zip created!')
print(f'   Size: {os.path.getsize("bert_model_download.zip") / 1e6:.1f} MB')

# Auto-download to your PC
files.download('bert_model_download.zip')
print('📥 Download started! Check your browser downloads.')


## 🎉 All Done! Next Steps

After `bert_model_download.zip` downloads to your PC:

### 1. Unzip it
Extract the zip file — you will get a `bert_model/` folder.

### 2. Place it in your project
```
fake_review_analyzer/
├── bert_model/          ← paste the unzipped folder here
│   ├── config.json
│   ├── pytorch_model.bin
│   ├── tokenizer files...
│   └── model_meta.pkl
├── app.py
├── generate_data.py
└── ...
```

### 3. Copy model_meta.pkl to root
Also copy `bert_model/model_meta.pkl` one level up to `fake_review_analyzer/`.

### 4. Launch the app
```bash
python -m streamlit run app.py
```

The sidebar will show **🤖 DistilBERT (Deep Learning)** — your model is live! 🚀
